# Giai đoạn 1 — Dữ liệu v2
Chọn sản phẩm chỉ từ train; bỏ hai tuần biên không đầy đủ; dùng chung split theo ngày. Notebook này ghi bộ mới vào `processed_data/v2`. Cài `requirements.txt` nếu môi trường chưa đủ thư viện.

In [ ]:
from pathlib import Path
import os
import sys

# Set this only if more than one source-code dataset is attached.
CODE_ROOT_OVERRIDE = None
if CODE_ROOT_OVERRIDE:
    CODE_ROOT = Path(CODE_ROOT_OVERRIDE).resolve()
else:
    candidates = {p.resolve() for p in [Path.cwd(), Path.cwd().parent] if (p/'src/config.py').is_file()}
    if Path('/kaggle/input').exists():
        for root, dirs, files in os.walk('/kaggle/input'):
            dirs[:] = [d for d in dirs if d not in {'images', '.git', '__pycache__'}]
            candidate = Path(root)
            if candidate.name == 'src' and 'config.py' in files and (candidate/'pipeline/train_baselines.py').is_file():
                candidates.add(candidate.parent.resolve())
    if len(candidates) != 1:
        raise ValueError(f'Set CODE_ROOT_OVERRIDE to the folder containing src. Candidates: {sorted(candidates)}')
    CODE_ROOT = next(iter(candidates))
if not (CODE_ROOT/'src/config.py').is_file():
    raise FileNotFoundError('The selected code folder is not source v2')
sys.path.insert(0, str(CODE_ROOT))
from src.utils.paths import resolve_data_dir, resolve_raw_dir, resolve_embeddings_dir
WORK_ROOT = Path('/kaggle/working') if Path('/kaggle/input').exists() else CODE_ROOT
print('Code:', CODE_ROOT, 'Output:', WORK_ROOT)

In [ ]:
RAW_DATA_OVERRIDE = None
RAW_DIR = resolve_raw_dir(CODE_ROOT, RAW_DATA_OVERRIDE)
PROCESSED_DIR = WORK_ROOT/'processed_data/v2'
from src.config import DataConfig
CONFIG = DataConfig()  # 80/12/12 complete weeks; train-only thresholds 20 weeks, 50 transactions; 8,000 items
print(CONFIG)

## 1. EDA
Đọc giao dịch theo từng khối để hạn chế RAM. Lượng bán quan sát là đại diện cho nhu cầu; không suy ra tồn kho từ số 0.

In [ ]:
from src.data.eda import run_eda
eda = run_eda(RAW_DIR, output_dir=WORK_ROOT/'eda/v2')


## 2. Tổng hợp và chọn mẫu
Lưu riêng `weekly_sales_all.csv` và `weekly_sales_selected.csv`. Giữ nguyên file toàn bộ sản phẩm để thử lại sampling.

In [ ]:
from src.pipeline.run_stage1_stage2 import run_stage1_stage2
summary = run_stage1_stage2(RAW_DIR, PROCESSED_DIR, CONFIG)
display(summary)


## 3. Xuất bộ input cho Kaggle
Upload toàn bộ archive này thành một dataset. Giữ nguyên nội dung các CSV để hash khớp. Không chỉ upload selected_articles.csv.

In [ ]:
import shutil
from IPython.display import FileLink, display
archive = shutil.make_archive(str(WORK_ROOT/'hm-stage1-v2'), 'zip', root_dir=PROCESSED_DIR)
display(FileLink(archive))


# Giai đoạn 2 — Trích xuất embedding từ ảnh thật
Gắn source code v2, bộ dữ liệu giai đoạn 1 v2 và H&M competition. Bật GPU và Internet cho lần tải encoder đầu tiên. Nếu thiếu thư viện, cài `requirements-extraction.txt`; không cần cài lại mỗi lần chạy.

In [ ]:
from pathlib import Path
import os
import sys

# Set this only if more than one source-code dataset is attached.
CODE_ROOT_OVERRIDE = None
if CODE_ROOT_OVERRIDE:
    CODE_ROOT = Path(CODE_ROOT_OVERRIDE).resolve()
else:
    candidates = {p.resolve() for p in [Path.cwd(), Path.cwd().parent] if (p/'src/config.py').is_file()}
    if Path('/kaggle/input').exists():
        for root, dirs, files in os.walk('/kaggle/input'):
            dirs[:] = [d for d in dirs if d not in {'images', '.git', '__pycache__'}]
            candidate = Path(root)
            if candidate.name == 'src' and 'config.py' in files and (candidate/'pipeline/train_baselines.py').is_file():
                candidates.add(candidate.parent.resolve())
    if len(candidates) != 1:
        raise ValueError(f'Set CODE_ROOT_OVERRIDE to the folder containing src. Candidates: {sorted(candidates)}')
    CODE_ROOT = next(iter(candidates))
if not (CODE_ROOT/'src/config.py').is_file():
    raise FileNotFoundError('The selected code folder is not source v2')
sys.path.insert(0, str(CODE_ROOT))
from src.utils.paths import resolve_data_dir, resolve_raw_dir, resolve_embeddings_dir
WORK_ROOT = Path('/kaggle/working') if Path('/kaggle/input').exists() else CODE_ROOT
print('Code:', CODE_ROOT, 'Output:', WORK_ROOT)


In [ ]:
DATA_DIR_OVERRIDE = None
RAW_DATA_OVERRIDE = None
IMAGES_DIR_OVERRIDE = None
DATA_DIR = resolve_data_dir(CODE_ROOT, DATA_DIR_OVERRIDE)
RAW_DIR = resolve_raw_dir(CODE_ROOT, RAW_DATA_OVERRIDE)
IMAGES_DIR = Path(IMAGES_DIR_OVERRIDE) if IMAGES_DIR_OVERRIDE else RAW_DIR/'images'
OUTPUT_DIR = WORK_ROOT/'embeddings/v2'
SELECTED_PATH = DATA_DIR/'selected_articles.csv'
BATCH_SIZE = 64
MAX_MISSING_IMAGE_FRACTION = 0.05  # cho phép một tỷ lệ nhỏ ảnh thiếu trong dữ liệu thật
IMAGE_REVISION = 'main'  # use the resolved commit in metadata to reproduce a later extraction
TEXT_REVISION = 'main'
import torch
print('Device:', 'cuda' if torch.cuda.is_available() else 'cpu')
print('Selection:', SELECTED_PATH, 'Images:', IMAGES_DIR, 'Output:', OUTPUT_DIR)


## 1. Xem ảnh đầu vào
Đây phải là ảnh sản phẩm thật, không phải ảnh đen thay thế. Pipeline tiếp theo sẽ kiểm tra toàn bộ ảnh và xuất image_audit.json.

In [ ]:
import pandas as pd
from IPython.display import display
from PIL import Image
from src.features.extract_embeddings import image_path
selected = pd.read_csv(SELECTED_PATH, dtype={'article_id': str})
for key in selected.article_id.sample(min(10, len(selected)), random_state=42):
    path = image_path(IMAGES_DIR, key)
    print(key, path)
    with Image.open(path) as source:
        preview = source.convert('RGB')
        preview.thumbnail((224, 224))
        display(preview)

## 2. Trích xuất và kiểm định
Ảnh 768d và text 384d được L2-normalize riêng. Hai encoder chưa có không gian ngữ nghĩa chung. Không tái tạo text theo công thức khác: dùng đúng text_for_embedding của Stage 1.

In [ ]:
from src.features.extract_embeddings import extract_embeddings
report = extract_embeddings(SELECTED_PATH, IMAGES_DIR, OUTPUT_DIR, BATCH_SIZE,
    max_missing_image_fraction=MAX_MISSING_IMAGE_FRACTION,
    image_revision=IMAGE_REVISION, text_revision=TEXT_REVISION)
display(report)

# Giai đoạn 3 — Baseline và kiểm tra thiếu modality
Dùng bộ dữ liệu và embedding v2 cùng hash. Nếu chưa trích xuất lại embedding, đặt MODELS thành danh sách sales-only như hướng dẫn ở cell cấu hình. Kết quả cũ không dùng để kết luận về lợi ích ảnh.

In [ ]:
from pathlib import Path
import os
import sys

# Set this only if more than one source-code dataset is attached.
CODE_ROOT_OVERRIDE = None
if CODE_ROOT_OVERRIDE:
    CODE_ROOT = Path(CODE_ROOT_OVERRIDE).resolve()
else:
    candidates = {p.resolve() for p in [Path.cwd(), Path.cwd().parent] if (p/'src/config.py').is_file()}
    if Path('/kaggle/input').exists():
        for root, dirs, files in os.walk('/kaggle/input'):
            dirs[:] = [d for d in dirs if d not in {'images', '.git', '__pycache__'}]
            candidate = Path(root)
            if candidate.name == 'src' and 'config.py' in files and (candidate/'pipeline/train_baselines.py').is_file():
                candidates.add(candidate.parent.resolve())
    if len(candidates) != 1:
        raise ValueError(f'Set CODE_ROOT_OVERRIDE to the folder containing src. Candidates: {sorted(candidates)}')
    CODE_ROOT = next(iter(candidates))
if not (CODE_ROOT/'src/config.py').is_file():
    raise FileNotFoundError('The selected code folder is not source v2')
sys.path.insert(0, str(CODE_ROOT))
from src.utils.paths import resolve_data_dir, resolve_raw_dir, resolve_embeddings_dir
WORK_ROOT = Path('/kaggle/working') if Path('/kaggle/input').exists() else CODE_ROOT
print('Code:', CODE_ROOT, 'Output:', WORK_ROOT)


In [ ]:
DATA_DIR_OVERRIDE = None
EMBEDDINGS_DIR_OVERRIDE = None
DATA_DIR = resolve_data_dir(CODE_ROOT, DATA_DIR_OVERRIDE)
MODELS = ['zero', 'last_value', 'moving_average', 'linear', 'mlp', 'lstm', 'transformer', 'multimodal_mlp']
# Before extraction: MODELS = ['zero', 'last_value', 'moving_average', 'linear', 'mlp', 'lstm', 'transformer']
EMBEDDINGS_DIR = resolve_embeddings_dir(CODE_ROOT, EMBEDDINGS_DIR_OVERRIDE) if 'multimodal_mlp' in MODELS else CODE_ROOT/'embeddings/v2'
SEEDS = [42, 43, 44]
EPOCHS = 30
PATIENCE = 5
BATCH_SIZE = 256
SEQ_LEN, PRED_LEN = 12, 2
TARGET_TRANSFORM = 'log1p'  # all learning models share this; use 'none' for a controlled raw-sales comparison
RETRAINED_ABLATION = True
# If running sales-only, set RETRAINED_ABLATION = False.
print('Data:', DATA_DIR, 'Embeddings:', EMBEDDINGS_DIR)


## 1. Chạy thực nghiệm
Cùng checkpoint full được đánh giá với 5 tình huống: full, mất ảnh, mất text, chỉ còn nửa lịch sử gần nhất, mất cả ảnh và text. Bảng retrained là thí nghiệm riêng, huấn luyện lại từng tổ hợp. Không diễn giải hai bảng như cùng một protocol.

### Lưu ý cell code dưới đây chỉ dùng trong kaggle này, không dùng trong local

In [ ]:
from pathlib import Path
import os

# 1. Xác định thư mục cha của dataset tùy theo môi trường
if os.path.exists('/kaggle/input'):
    # Chạy trên Kaggle: đường dẫn dataset mount vào
    BASE_EMB = Path('/kaggle/input/datasets/alexnguyen147/hm-embeddings-8k')
    
    # Một số trường hợp Kaggle slug đường dẫn không có chữ 'datasets/'
    if not BASE_EMB.exists():
        BASE_EMB = Path('/kaggle/input/hm-embeddings-8k')
else:
    # Chạy ở Local: đặt đường dẫn thư mục embeddings ở máy bạn
    BASE_EMB = Path('embeddings')  # hoặc Path('data/embeddings') tùy theo máy bạn

# 2. Tự động kiểm tra: nếu có thư mục con 'embeddings' thì trỏ vào bên trong
if (BASE_EMB / 'embeddings').exists():
    EMBEDDINGS_DIR = BASE_EMB / 'embeddings'
else:
    EMBEDDINGS_DIR = BASE_EMB

print(f"-> Using EMBEDDINGS_DIR: {EMBEDDINGS_DIR.resolve()}")
print(f"-> Metadata exists: {(EMBEDDINGS_DIR / 'metadata.json').exists()}")

In [ ]:
from src.pipeline.train_baselines import run_all_baselines
RUN_DIR = run_all_baselines(epochs=EPOCHS, patience=PATIENCE, seeds=SEEDS,
    batch_size=BATCH_SIZE, processed_data_dir=DATA_DIR, embeddings_dir=EMBEDDINGS_DIR,
    output_dir=WORK_ROOT/'results/v2', models=MODELS, seq_len=SEQ_LEN, pred_len=PRED_LEN,
    target_transform=TARGET_TRANSFORM, retrained_ablation=RETRAINED_ABLATION)


## 2. Kết quả và độ suy giảm
MAE/RMSE ở đơn vị lượt bán. wMAPE/sMAPE tính bằng %. Khi tổng actual bằng 0, wMAPE không xác định và được lưu trống. Không dùng test để chọn siêu tham số.

In [ ]:
import pandas as pd
from IPython.display import display
display(pd.read_csv(RUN_DIR/'baseline_results.csv'))
display(pd.read_csv(RUN_DIR/'seed_summary.csv'))
if (RUN_DIR/'robustness_comparison.csv').exists():
    display(pd.read_csv(RUN_DIR/'robustness_comparison.csv'))
display(pd.read_csv(RUN_DIR/'detailed_metrics.csv').head(30))


## 3. Lưu kết quả
Mỗi run có cấu hình, hash dữ liệu và source, checkpoint, learning curves dạng CSV, dự đoán theo sản phẩm/tuần/horizon, metric theo nhóm và summary seed.

In [ ]:
import shutil
from IPython.display import FileLink
archive = shutil.make_archive(str(WORK_ROOT/('baseline-results-'+RUN_DIR.name)), 'zip', root_dir=RUN_DIR)
display(FileLink(archive))
